[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lmassaron/finetuning/blob/main/chapter_05/listing_5.1-5.7.ipynb)

In [ ]:
import sys
if "google.colab" in sys.modules:
    get_ipython().system("pip install -q -U transformers datasets peft accelerate optimum trl bitsandbytes")

### Listing 5.1: Checking GPU memory usage with PyTorch

In [14]:
import torch
def check_used_memory():
   device = torch.device("cuda")
   props = torch.cuda.get_device_properties(device)
   total = props.total_memory / 1024**3
   used = torch.cuda.memory_allocated(device) / 1024**3
   reserved = torch.cuda.memory_reserved(device) / 1024**3
   
   print(f"Total GPU memory : {total:.2f} GB")
   print(f"Used memory      : {used:.2f} GB")
   print(f"Reserved memory  : {reserved:.2f} GB")
check_used_memory()

Total GPU memory : 121.69 GB
Used memory      : 1.95 GB
Reserved memory  : 2.00 GB


### Listing 5.2: Configuring LoRA for Llama-3-8B using PEFT

In [15]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model_id = "Qwen/Qwen2.5-0.5B"
base_model = AutoModelForCausalLM.from_pretrained(model_id)
peft_config = LoraConfig(
   r=64,
   lora_alpha=16,
   lora_dropout=0,
   bias="none",
   task_type="CAUSAL_LM",
   target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(base_model, peft_config)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

### Listing 5.3: Enabling and disabling LoRA adapters on the fly

In [16]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig

base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B")
peft_config = LoraConfig(
   r=8,
   target_modules=["q_proj", "v_proj"],
   task_type="CAUSAL_LM"
)

base_model.add_adapter(peft_config)

# You can even enable/disable the adapters on the fly
base_model.disable_adapters()
base_model.enable_adapters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

### Listing 5.4: Setting up a LoRA training loop using SFTTrainer

In [17]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B")
my_dataset = Dataset.from_dict({"text": [
    "Hello world, this is a LoRA training example.", 
    "Fine-tuning LLMs with PEFT and TRL."
    ]})

peft_config = LoraConfig(
   r=8, 
   target_modules=["q_proj", "v_proj"], 
   task_type="CAUSAL_LM",
)
training_args = SFTConfig(
   dataset_text_field="text",
)
trainer = SFTTrainer(
   model=base_model,
   args=training_args,
   peft_config=peft_config,
   train_dataset=my_dataset,
)

trainer.train()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.
[datasets.arrow_dataset|WARNING]num_proc must be <= 2. Reducing num_proc to 2 for dataset of size 2.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2 | Num Epochs = 3 | Total steps = 3
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 494,032,768 of 494,032,768 (100.00% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


AttributeError: 'Qwen2Attention' object has no attribute 'apply_qkv'

### Listing 5.5: Setting up QLoRA (Quantized LoRA) using BitsAndBytes and PEFT

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, \
   BitsAndBytesConfig

from peft import LoraConfig, get_peft_model

quant_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.bfloat16
)

model_id = "Qwen/Qwen2.5-0.5B"
base_model = AutoModelForCausalLM.from_pretrained(
   model_id, device_map="auto", quantization_config=quant_config
)

lora_config = LoraConfig(
   r=16, lora_alpha=32, target_modules="all-linear",
   lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

### Listing 5.6: Fast model loading and LoRA configuration using Unsloth

In [ ]:
from unsloth import FastLanguageModel

model_id = "Qwen/Qwen2.5-0.5B"
model, tokenizer = FastLanguageModel.from_pretrained(
   model_name=model_id,
   max_seq_length=2048,
   load_in_4bit=True,
   full_finetuning=False,
)
model = FastLanguageModel.get_peft_model(
   model,
   r=32,
   target_modules=["q_proj", "k_proj", "v_proj",
                    "o_proj", "gate_proj", "up_proj", "down_proj"],
   lora_alpha=32,
   lora_dropout=0,
   bias="none",
   use_gradient_checkpointing="unsloth",
   random_state=0,
   use_rslora=False,
   loftq_config=None,
)

/home/lmassaron/code/finetuning/chapter_06/.venv_ch06/lib/python3.12/site-packages/unsloth/__init__.py:1432: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Unsloth 2026.7.6 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


### Listing 5.7: Preparing a pre-quantized GPTQ model for LoRA training

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

model_id = "Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4"
model = AutoModelForCausalLM.from_pretrained(
   model_id,
   device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
   r=16,
   lora_alpha=32,
   target_modules=["q_proj", "v_proj"],
   lora_dropout=0.05,
   bias="none",
   task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

ImportError: Loading a GPTQ quantized model requires optimum (`pip install optimum`)